# v16.3 — replikasi detektor AF 2 kelas D-1 (ECGFounder, ambang −1) di CinC 2017, MIT-BIH, LTAFDB

Rancangan dikunci sebelum sinyal dibaca: [RANCANGAN.md](RANCANGAN.md) (sha256 di `../logs/v16.3_praregistrasi.log`). Logika di
`replikasi.py`. Ketiga dataset belum pernah dipakai SIHEDAF; tidak ada yang dilatih atau disetel. Hasil per rekaman disimpan di
`../cache/v16.3/` (rekaman yang sudah ada dilewati).

| Sistem | Keputusan AF |
|---|---|
| **D-1 (primer)** | ECGFounder zero-shot: log((pAF + pAFL) / pN) > −1 |
| **v3.1 paket (referensi)**, v1 paket | argmax paket pada RR window, AFIB + AFL digabung |
| D-0 (v16.1) | ... > 0 |

Kriteria per dataset: H1 Macro-F1 2 kelas dan H2 sensitivitas D-1 bukan "lebih buruk" dari v3.1. Replikasi berhasil bila keduanya lolos
di ketiga dataset.

In [1]:
import json, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import replikasi as A
from replikasi import Ev


def dl(v):
    return f"Δ {v[0]:+.3f} [{v[1]:+.3f}…{v[2]:+.3f}], P = {v[3]:.3f} → {Ev.vonis(v[3])}"


al = None
for ds in A.DATASET:
    al = A.jalankan(ds, al)
data, jumlah = {}, {}
for ds in A.DATASET:
    data[ds], jumlah[ds] = A.muat(ds)
    print(ds, "| rekaman:", len(A.rekaman(ds)), "| window lolos:", len(data[ds]["y2"]), "| AF:", int(data[ds]["y2"].sum()),
          "| unit bootstrap:", np.unique(data[ds]["subjek"]).size)
pd.DataFrame({ds: {r: f"{n[1]:,} / {n[0]:,}" for r, n in j.items()} for ds, j in jumlah.items()}).fillna("").rename_axis("window lolos / berlabel")

CinC 2017 | rekaman: 8249 | window lolos: 43465 | AF: 3898 | unit bootstrap: 8172
MIT-BIH | rekaman: 48 | window lolos: 14152 | AF: 1514 | unit bootstrap: 46


LTAFDB | rekaman: 84 | window lolos: 1272328 | AF: 706582 | unit bootstrap: 84


,CinC 2017,MIT-BIH,LTAFDB
window lolos / berlabel,,,
N,"26,227 / 27,052",,
A,"3,898 / 4,079",,
O,"13,340 / 13,943",,
(N,,"11,726 / 11,882","537,559 / 563,522"
(B,,233 / 246,437 / 443
(T,,94 / 96,147 / 152
(IVR,,21 / 22,
(NOD,,5 / 5,
(AFIB,,"1,416 / 1,433","706,582 / 731,303"


## 1. Hasil utama per dataset (H1, H2)

In [2]:
hasil = {ds: A.evaluasi(ds, data[ds]) for ds in A.DATASET}
for ds, h in hasil.items():
    print(f"== {ds}", h["n"])
    display(pd.DataFrame({k: {m: f"{v[m][0]:.3f} [{v[m][1]:.2f}–{v[m][2]:.2f}]" for m in ("Macro-F1", "Se", "Sp", "PPV")} for k, v in h["titik"].items()}).T
            .rename_axis(ds))
    display(pd.DataFrame({k: {m: dl(v[m]) for m in ("Macro-F1", "Se", "Sp", "PPV")} for k, v in h["banding"].items()}).T
            .rename_axis(f"{ds}: vs v3.1 (kecuali baris terakhir)"))

== CinC 2017 {'window': 43465, 'af': 3898, 'non_af': 39567, 'unit': 8172, 'unit_af': 740}


,Macro-F1,Se,Sp,PPV
CinC 2017,,,,
v3.1 paket,0.757 [0.74–0.77],0.884 [0.87–0.90],0.886 [0.88–0.89],0.433 [0.41–0.46]
D-1,0.867 [0.86–0.88],0.830 [0.80–0.85],0.965 [0.96–0.97],0.702 [0.67–0.73]
D-0,0.881 [0.87–0.89],0.759 [0.73–0.78],0.982 [0.98–0.98],0.807 [0.78–0.83]
v1 paket,0.726 [0.71–0.74],0.896 [0.88–0.91],0.856 [0.85–0.86],0.381 [0.36–0.40]


,Macro-F1,Se,Sp,PPV
CinC 2017: vs v3.1 (kecuali baris terakhir),,,,
D-1,"Δ +0.110 [+0.099…+0.121], P = 1.000 → lebih baik","Δ -0.054 [-0.074…-0.035], P = 0.000 → lebih buruk","Δ +0.079 [+0.074…+0.085], P = 1.000 → lebih baik","Δ +0.269 [+0.248…+0.289], P = 1.000 → lebih baik"
D-0,"Δ +0.123 [+0.111…+0.135], P = 1.000 → lebih baik","Δ -0.125 [-0.147…-0.104], P = 0.000 → lebih buruk","Δ +0.096 [+0.091…+0.102], P = 1.000 → lebih baik","Δ +0.375 [+0.352…+0.396], P = 1.000 → lebih baik"
v1 paket,"Δ -0.032 [-0.036…-0.027], P = 0.000 → lebih buruk","Δ +0.012 [+0.003…+0.021], P = 0.995 → lebih baik","Δ -0.029 [-0.033…-0.026], P = 0.000 → lebih buruk","Δ -0.052 [-0.059…-0.045], P = 0.000 → lebih buruk"
D-1 vs D-0,"Δ -0.013 [-0.019…-0.008], P = 0.000 → lebih buruk","Δ +0.071 [+0.061…+0.082], P = 1.000 → lebih baik","Δ -0.017 [-0.019…-0.015], P = 0.000 → lebih buruk","Δ -0.105 [-0.117…-0.094], P = 0.000 → lebih buruk"


== MIT-BIH {'window': 14152, 'af': 1514, 'non_af': 12638, 'unit': 46, 'unit_af': 7}


,Macro-F1,Se,Sp,PPV
MIT-BIH,,,,
v3.1 paket,0.754 [0.60–0.84],0.916 [0.87–0.96],0.857 [0.79–0.91],0.435 [0.17–0.64]
D-1,0.967 [0.90–0.99],0.983 [0.97–1.00],0.987 [0.98–0.99],0.902 [0.69–0.97]
D-0,0.979 [0.94–0.99],0.968 [0.93–0.99],0.995 [0.99–1.00],0.958 [0.83–0.99]
v1 paket,0.714 [0.56–0.81],0.938 [0.87–0.99],0.812 [0.74–0.87],0.374 [0.14–0.58]


,Macro-F1,Se,Sp,PPV
MIT-BIH: vs v3.1 (kecuali baris terakhir),,,,
D-1,"Δ +0.213 [+0.136…+0.307], P = 1.000 → lebih baik","Δ +0.067 [+0.011…+0.122], P = 0.997 → lebih baik","Δ +0.130 [+0.076…+0.194], P = 1.000 → lebih baik","Δ +0.467 [+0.310…+0.587], P = 1.000 → lebih baik"
D-0,"Δ +0.225 [+0.139…+0.348], P = 1.000 → lebih baik","Δ +0.052 [-0.019…+0.114], P = 0.935 → tidak te...","Δ +0.137 [+0.082…+0.203], P = 1.000 → lebih baik","Δ +0.522 [+0.340…+0.696], P = 1.000 → lebih baik"
v1 paket,"Δ -0.040 [-0.072…-0.014], P = 0.001 → lebih buruk","Δ +0.022 [-0.008…+0.044], P = 0.928 → tidak te...","Δ -0.045 [-0.074…-0.020], P = 0.000 → lebih buruk","Δ -0.061 [-0.115…-0.019], P = 0.000 → lebih buruk"
D-1 vs D-0,"Δ -0.012 [-0.045…-0.001], P = 0.018 → lebih buruk","Δ +0.016 [+0.004…+0.034], P = 0.994 → lebih baik","Δ -0.008 [-0.013…-0.004], P = 0.000 → lebih buruk","Δ -0.055 [-0.148…-0.022], P = 0.000 → lebih buruk"


== LTAFDB {'window': 1272328, 'af': 706582, 'non_af': 565746, 'unit': 84, 'unit_af': 83}


,Macro-F1,Se,Sp,PPV
LTAFDB,,,,
v3.1 paket,0.933 [0.91–0.95],0.948 [0.93–0.97],0.915 [0.88–0.95],0.933 [0.90–0.96]
D-1,0.967 [0.95–0.98],0.974 [0.96–0.99],0.960 [0.93–0.98],0.968 [0.94–0.99]
D-0,0.963 [0.95–0.98],0.945 [0.92–0.96],0.985 [0.97–1.00],0.987 [0.97–1.00]
v1 paket,0.930 [0.91–0.95],0.965 [0.95–0.98],0.890 [0.85–0.93],0.916 [0.87–0.95]


,Macro-F1,Se,Sp,PPV
LTAFDB: vs v3.1 (kecuali baris terakhir),,,,
D-1,"Δ +0.035 [+0.019…+0.052], P = 1.000 → lebih baik","Δ +0.025 [+0.013…+0.038], P = 1.000 → lebih baik","Δ +0.045 [+0.013…+0.078], P = 0.998 → lebih baik","Δ +0.035 [+0.011…+0.065], P = 0.998 → lebih baik"
D-0,"Δ +0.030 [+0.015…+0.047], P = 1.000 → lebih baik","Δ -0.003 [-0.021…+0.013], P = 0.354 → tidak te...","Δ +0.070 [+0.044…+0.101], P = 1.000 → lebih baik","Δ +0.054 [+0.032…+0.087], P = 1.000 → lebih baik"
v1 paket,"Δ -0.003 [-0.011…+0.005], P = 0.263 → tidak te...","Δ +0.016 [+0.010…+0.023], P = 1.000 → lebih baik","Δ -0.025 [-0.038…-0.014], P = 0.000 → lebih buruk","Δ -0.017 [-0.028…-0.008], P = 0.000 → lebih buruk"
D-1 vs D-0,"Δ +0.005 [-0.006…+0.014], P = 0.810 → tidak te...","Δ +0.028 [+0.019…+0.039], P = 1.000 → lebih baik","Δ -0.025 [-0.043…-0.011], P = 0.000 → lebih buruk","Δ -0.019 [-0.036…-0.007], P = 0.000 → lebih buruk"


## 2. Alarm per jenis ritme, per pita detak, per subjek, episode

In [3]:
for ds, h in hasil.items():
    display(pd.DataFrame(h["ritme"]).T.round(3).rename_axis(f"{ds}: proporsi window teralarm AF per ritme"))
    display(pd.DataFrame({k: {b: (f"{x['alarm']*100:.1f}% (n={x['n']})" if x["alarm"] is not None else "–") for b, x in v.items()} for k, v in h["pita"].items()}).T
            .rename_axis(f"{ds}: alarm non-AF per pita detak"))
    print(ds, "| unit dengan sensitivitas D-1 turun > 0,20:", h["se_turun_0_20"])
    if "episode" in h:
        display(pd.DataFrame(h["episode"]).T.round(3).rename_axis(f"{ds}: episode AF (EC57)"))

,n,v3.1 paket,v1 paket,D-0,D-1
CinC 2017: proporsi window teralarm AF per ritme,,,,,
A,3898.0,0.884,0.896,0.759,0.830
N,26227.0,0.069,0.096,0.007,0.015
O,13340.0,0.202,0.238,0.040,0.074


,< 100,100–110,110–120,≥ 120
CinC 2017: alarm non-AF per pita detak,,,,
v3.1 paket,9.4% (n=37663),17.9% (n=955),62.7% (n=386),98.2% (n=563)
v1 paket,12.1% (n=37663),25.8% (n=955),83.9% (n=386),98.0% (n=563)
D-0,1.6% (n=37663),5.5% (n=955),3.1% (n=386),6.9% (n=563)
D-1,3.2% (n=37663),8.6% (n=955),6.0% (n=386),12.1% (n=563)


CinC 2017 | unit dengan sensitivitas D-1 turun > 0,20: {'A00004': 0.4, 'A00087': 0.6, 'A00282': 1.0, 'A00321': 1.0, 'A00404': 0.5, 'A00422': 0.6, 'A00473': 0.2, 'A00493': 0.333, 'A00691': 0.4, 'A00706': 0.2, 'A00776': 1.0, 'A00902': 0.4, 'A00932': 1.0, 'A01019': 0.4, 'A01079': 0.4, 'A01129': 0.4, 'A01171': 0.6, 'A01256': 0.4, 'A01302': 0.273, 'A01359': 0.455, 'A01376': 0.6, 'A01685': 0.444, 'A01693': 0.8, 'A01784': 0.6, 'A01824': 0.6, 'A01831': 0.6, 'A01869': 0.4, 'A01892': 0.8, 'A02051': 0.4, 'A02102': 0.545, 'A02325': 0.4, 'A02474': 0.4, 'A02557': 0.4, 'A02584': 0.4, 'A02587': 0.4, 'A02613': 0.909, 'A02644': 0.6, 'A02695': 1.0, 'A02805': 0.5, 'A02856': 0.5, 'A03063': 0.4, 'A03078': 1.0, 'A03221': 0.7, 'A03397': 0.8, 'A03569': 1.0, 'A03584': 0.333, 'A03606': 0.25, 'A03760': 0.6, 'A03838': 0.4, 'A03862': 0.6, 'A04128': 0.2, 'A04201': 0.8, 'A04230': 0.6, 'A04348': 0.8, 'A04454': 0.8, 'A04493': 1.0, 'A04591': 0.4, 'A04604': 0.2, 'A04612': 0.25, 'A04638': 0.6, 'A04675': 1.0, 'A04712': 1.0

,n,v3.1 paket,v1 paket,D-0,D-1
MIT-BIH: proporsi window teralarm AF per ritme,,,,,
(AB,13.0,0.462,0.462,0.154,0.385
(AFIB,1416.0,0.916,0.939,0.971,0.987
(AFL,98.0,0.918,0.918,0.918,0.939
(B,233.0,0.571,0.498,0.026,0.052
(BII,130.0,0.000,0.000,0.000,0.000
(IVR,21.0,0.048,0.048,0.190,0.524
(N,11726.0,0.127,0.179,0.003,0.009
(NOD,5.0,0.000,0.000,0.000,0.200
(PREX,52.0,0.058,0.019,0.000,0.000


,< 100,100–110,110–120,≥ 120
MIT-BIH: alarm non-AF per pita detak,,,,
v3.1 paket,12.0% (n=11625),19.0% (n=648),76.7% (n=347),100.0% (n=18)
v1 paket,15.7% (n=11625),34.9% (n=648),88.8% (n=347),100.0% (n=18)
D-0,0.4% (n=11625),1.5% (n=648),1.7% (n=347),0.0% (n=18)
D-1,1.1% (n=11625),2.8% (n=648),4.6% (n=347),11.1% (n=18)


MIT-BIH | unit dengan sensitivitas D-1 turun > 0,20: {}


,se_episode,ppv_episode,se_durasi,ppv_durasi,n_ref,n_det,palsu_per_jam
MIT-BIH: episode AF (EC57),,,,,,,
v3.1 paket,0.946,0.174,0.916,0.435,74.0,886.0,37.241
v1 paket,0.932,0.112,0.938,0.374,74.0,1065.0,48.129
D-0,0.959,0.705,0.968,0.958,74.0,139.0,2.086
D-1,0.986,0.464,0.983,0.902,74.0,181.0,4.935


,n,v3.1 paket,v1 paket,D-0,D-1
LTAFDB: proporsi window teralarm AF per ritme,,,,,
(AB,1503.0,0.579,0.611,0.023,0.094
(AFIB,706582.0,0.948,0.965,0.945,0.974
(B,437.0,0.515,0.348,0.007,0.023
(N,537559.0,0.085,0.113,0.013,0.028
(SBR,25501.0,0.042,0.017,0.054,0.279
(SVTA,599.0,0.265,0.337,0.047,0.100
(T,147.0,0.544,0.578,0.184,0.238


,< 100,100–110,110–120,≥ 120
LTAFDB: alarm non-AF per pita detak,,,,
v3.1 paket,7.1% (n=548935),29.7% (n=9453),73.2% (n=4244),96.7% (n=3114)
v1 paket,9.4% (n=548935),40.5% (n=9453),89.0% (n=4244),96.6% (n=3114)
D-0,1.3% (n=548935),5.7% (n=9453),8.1% (n=4244),6.7% (n=3114)
D-1,3.6% (n=548935),11.8% (n=9453),20.8% (n=4244),21.2% (n=3114)


LTAFDB | unit dengan sensitivitas D-1 turun > 0,20: {'116': 0.356, '16': 0.274}


,se_episode,ppv_episode,se_durasi,ppv_durasi,n_ref,n_det,palsu_per_jam
LTAFDB: episode AF (EC57),,,,,,,
v3.1 paket,0.971,0.577,0.948,0.933,8902.0,52322.0,12.514
v1 paket,0.962,0.449,0.965,0.916,8902.0,50189.0,15.643
D-0,0.802,0.873,0.945,0.987,8902.0,29052.0,2.094
D-1,0.900,0.610,0.974,0.968,8902.0,26004.0,5.740


In [4]:
kriteria = {ds: h["kriteria"] for ds, h in hasil.items()}
lolos = sum(k["H1_macro_f1"] and k["H2_sensitivitas"] for k in kriteria.values())
vonis_replikasi = "berhasil" if lolos == len(A.DATASET) else ("sebagian" if lolos else "gagal")
ring = {"hasil": hasil, "jumlah": jumlah, "kriteria": kriteria, "lolos": lolos, "vonis_replikasi": vonis_replikasi, "ambang": A.AMBANG,
        "lingkungan": Ev._lingkungan()}
A.CACHE.mkdir(parents=True, exist_ok=True)
(A.CACHE / "ringkasan.json").write_text(json.dumps(ring, indent=1, default=float, ensure_ascii=False))
print("kriteria:", kriteria)
print("vonis replikasi:", vonis_replikasi, f"({lolos}/{len(A.DATASET)} dataset)")
print("tersimpan:", A.CACHE / "ringkasan.json")

kriteria: {'CinC 2017': {'H1_macro_f1': True, 'H2_sensitivitas': False}, 'MIT-BIH': {'H1_macro_f1': True, 'H2_sensitivitas': True}, 'LTAFDB': {'H1_macro_f1': True, 'H2_sensitivitas': True}}
vonis replikasi: sebagian (2/3 dataset)
tersimpan: /mnt/HDD/03_Akademik_dan_Tugas/WrapSihedaf/cache/v16.3/ringkasan.json


## 3. Kesimpulan

Dijalankan penuh 6 November 2026 di PC (RTX 5060; hasil per rekaman ±40 menit dengan dua proses, notebook ±1 menit dari cache). Vonis
yang dikunci: **replikasi sebagian (2 dari 3 dataset)**. Tidak ada ambang yang diubah.

- **MIT-BIH** (46 subjek, 7 ber-AF): D-1 0,967 vs v3.1 0,754 (lebih baik); sensitivitas 0,983 vs 0,916 (lebih baik); spesifisitas 0,987 vs
  0,857. Episode AF palsu 37,2 → 4,9 per jam. Lolos.
- **LTAFDB** (84 subjek, 1,27 juta window): D-1 0,967 vs 0,933 (lebih baik); sensitivitas 0,974 vs 0,948 (lebih baik); spesifisitas 0,960
  vs 0,915. Episode AF palsu 12,5 → 5,7 per jam, tetapi Se episode 0,97 → 0,90 (episode pendek lebih sering terlewat). Lolos.
- **CinC 2017** (EKG genggam satu lead, 8.249 rekaman): D-1 0,867 vs 0,757 (lebih baik), tetapi **sensitivitas 0,830 vs 0,884 (lebih
  buruk, P = 0,000)** → H2 gagal. Spesifisitas 0,965 vs 0,886; PPV 0,70 vs 0,43; ritme lain (`O`) dialarmkan 7,4% vs 20,2%.
- **Pola lintas dataset:** pada data genggam/wearable satu lead (CinC 2017, dan Icentia11k di v16.2) D-1 kehilangan ±3–5 poin
  sensitivitas; pada Holter (CPSC 2021, MIT-BIH, LTAFDB) sensitivitasnya setara atau lebih baik. Spesifisitas lebih baik di semua data.
- **Kelemahan baru:** di LTAFDB, window sinus bradikardia dialarmkan 27,9% oleh D-1 (v3.1 4,2%; D-0 5,4%), dan beberapa aritmia
  ventrikel/supraventrikel di MIT-BIH ikut dialarmkan (jumlah window kecil).
- **Status:** D-1 tetap kandidat detektor AF 2 kelas untuk uji perangkat, dengan catatan bahwa sensitivitas di perangkat wearable
  harus menjadi kriteria utama uji itu. Paket v3.1 tetap.